Created access to google drive

In [2]:
# Mount Google Drive (to save your model permanently)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Uploaded the Kaggle API (legacy) key, and then, unzipped the folder

In [3]:
# Upload kaggle.json and download dataset
from google.colab import files
files.upload()  # select your kaggle.json when prompted

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and unzip dataset
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip
!ls

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
100% 2.04G/2.04G [00:23<00:00, 91.5MB/s]

 drive	      'plantvillage dataset'	  sample_data
 kaggle.json   plantvillage-dataset.zip


Turns out there was a folder in a folder, so inside platvillage-dataset.zip, we need to open 'plantvillage dataset'

In [4]:
!unzip -q plantvillage-dataset.zip -d plantvillage

In [5]:
!ls 'plantvillage dataset'/

color  grayscale  segmented


In [6]:
!ls 'plantvillage dataset'/color/ | head -20

Apple___Apple_scab
Apple___Black_rot
Apple___Cedar_apple_rust
Apple___healthy
Blueberry___healthy
Cherry_(including_sour)___healthy
Cherry_(including_sour)___Powdery_mildew
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
Corn_(maize)___Common_rust_
Corn_(maize)___healthy
Corn_(maize)___Northern_Leaf_Blight
Grape___Black_rot
Grape___Esca_(Black_Measles)
Grape___healthy
Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
Orange___Haunglongbing_(Citrus_greening)
Peach___Bacterial_spot
Peach___healthy
Pepper,_bell___Bacterial_spot
Pepper,_bell___healthy


In [7]:
import torch
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, random_split

# Path to dataset
DATA_DIR = '/content/plantvillage dataset/color'

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load full dataset
dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

# Split BEFORE any fitting — 70/15/15
train_size = int(0.70 * len(dataset))
val_size   = int(0.15 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

# Dataloaders
train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False, num_workers=2)

# Confirm
print(f"Classes: {len(dataset.classes)}")
print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
print(f"Sample classes: {dataset.classes[:5]}")

Classes: 38
Train: 38013 | Val: 8145 | Test: 8147
Sample classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


In [8]:
import torch.nn as nn

class PlantCNN(nn.Module):
    def __init__(self, num_classes=38):
        super(PlantCNN, self).__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 112x112

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 56x56

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 28x28

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 14x14
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Initialize model and move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PlantCNN(num_classes=38).to(device)

print(f"Using device: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Using device: cuda
Total parameters: 26,099,494


In [9]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(train_set)

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(val_set)

# Training loop
EPOCHS = 40
best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)
    scheduler.step()

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/plant_best.pth')

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.3f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.3f} Acc: {val_acc:.3f} | "
          f"Best Val: {best_val_acc:.3f}")

Epoch 01/40 | Train Loss: 3.366 Acc: 0.190 | Val Loss: 2.405 Acc: 0.296 | Best Val: 0.296
Epoch 02/40 | Train Loss: 2.765 Acc: 0.230 | Val Loss: 2.412 Acc: 0.356 | Best Val: 0.356
Epoch 03/40 | Train Loss: 2.685 Acc: 0.246 | Val Loss: 2.154 Acc: 0.413 | Best Val: 0.413
Epoch 04/40 | Train Loss: 2.616 Acc: 0.259 | Val Loss: 2.045 Acc: 0.434 | Best Val: 0.434
Epoch 05/40 | Train Loss: 2.516 Acc: 0.288 | Val Loss: 1.830 Acc: 0.501 | Best Val: 0.501
Epoch 06/40 | Train Loss: 2.206 Acc: 0.367 | Val Loss: 1.495 Acc: 0.595 | Best Val: 0.595
Epoch 07/40 | Train Loss: 2.003 Acc: 0.417 | Val Loss: 1.224 Acc: 0.652 | Best Val: 0.652
Epoch 08/40 | Train Loss: 1.757 Acc: 0.483 | Val Loss: 0.975 Acc: 0.711 | Best Val: 0.711
Epoch 09/40 | Train Loss: 1.563 Acc: 0.534 | Val Loss: 0.830 Acc: 0.763 | Best Val: 0.763
Epoch 10/40 | Train Loss: 1.400 Acc: 0.578 | Val Loss: 0.656 Acc: 0.815 | Best Val: 0.815
Epoch 11/40 | Train Loss: 1.225 Acc: 0.624 | Val Loss: 0.545 Acc: 0.846 | Best Val: 0.846
Epoch 12/4

In [13]:
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds,
      target_names=dataset.classes))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.92      0.76      0.83       103
                                 Apple___Black_rot       0.99      0.91      0.95        93
                          Apple___Cedar_apple_rust       1.00      0.90      0.95        30
                                   Apple___healthy       0.93      0.97      0.95       238
                               Blueberry___healthy       0.97      0.99      0.98       213
          Cherry_(including_sour)___Powdery_mildew       0.99      0.96      0.97       168
                 Cherry_(including_sour)___healthy       0.93      1.00      0.96       120
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.80      0.29      0.42        83
                       Corn_(maize)___Common_rust_       0.99      0.99      0.99       168
               Corn_(maize)___Northern_Leaf_Blight       0.71      0.96      0.

In [15]:
import torch
!pip install onnxscript -q
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    model,
    dummy_input,
    '/content/drive/MyDrive/plant_model.onnx',
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}}
)

print("ONNX export complete — saved to Google Drive")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 19.4 MB/s eta 0:00:00


/tmp/ipykernel_7525/1490323877.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `PlantCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `PlantCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX export complete — saved to Google Drive


In [18]:
import os

path = '/content/drive/MyDrive/plant_model.onnx'
exists = os.path.exists(path)
print(f"File exists: {exists}")

# Also list what's in your Drive root
!ls /content/drive/MyDrive/

File exists: True
'$$$ YO SHRINCAT READ THIS $$$ .gdoc'
'1545786781345_DASASHREE MODEL QUESTION PAPERS- 2.pdf'
'1.8.2 practice assignmenet.gdoc'
 1A0722DB-5870-42F7-802A-308A68CAB40B.JPG
'2025 Security Briefing Attestation.docx'
 23-812.records.xlsx
'492 Induvidual Presentation picture ex.gdoc'
 63A85A63-7117-480B-819F-9C1AD826FC4C.JPG
 709F86EE-AF74-4040-8252-57C65E1DA133.JPG
 7CCSSFinal_Exam_Review.pdf.gdoc
'9 5 24 resume.gdoc'
 97CC9826-D3A5-4AC2-A173-1495AFEC5E94.JPG
'AI Music Quiz (before & after) (Responses).gsheet'
'AI Music Quiz (before using the app).gform'
'AI Transforms Creative Workflows.gslides'
'AKPsi fall 2024 questions.gdoc'
'Algebra Home Work Sheet-1.pdf'
'Algebra Home Work Sheet-2 (1).gdoc'
'Algebra Home Work Sheet-2 (1).pdf'
'Alumni Information Google Form.gform'
'Annotated Bib - Vishruth Gonur.gdoc'
'ANUSHMAN & KEON FA'\'' 25 [DAY 2] First-Round Interview.gdoc'
'Appendix C - Pictures.gdoc'
'Assignment 1 - Vishruth Gonur.gdoc'
'Assignment 2 IS492 Video Link.gdoc'
'Ba

In [20]:
import torch

model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)

torch.onnx.export(
    model,
    dummy_input,
    '/content/drive/MyDrive/plant_model.onnx',
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}}
)

size = os.path.getsize('/content/drive/MyDrive/plant_model.onnx')
print(f"Model size: {size / 1024 / 1024:.1f} MB")

/tmp/ipykernel_7525/3260728780.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0531 20:55:42.384000 7525 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `PlantCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `PlantCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model size: 0.0 MB


In [21]:
import torch
import os

model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Force legacy exporter
torch.onnx.export(
    model,
    dummy_input,
    '/content/drive/MyDrive/plant_model_v2.onnx',
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamo=False  # force legacy exporter
)

size = os.path.getsize('/content/drive/MyDrive/plant_model_v2.onnx')
print(f"Model size: {size / 1024 / 1024:.1f} MB")

/tmp/ipykernel_7525/1030359176.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Model size: 99.6 MB


In [22]:
import json

# Save class labels to Drive
labels = {i: name for i, name in enumerate(dataset.classes)}

with open('/content/drive/MyDrive/labels.json', 'w') as f:
    json.dump(labels, f, indent=2)

print("Labels saved. First 5:")
for i in range(5):
    print(f"  {i}: {labels[i]}")

Labels saved. First 5:
  0: Apple___Apple_scab
  1: Apple___Black_rot
  2: Apple___Cedar_apple_rust
  3: Apple___healthy
  4: Blueberry___healthy
